<a href="https://colab.research.google.com/github/vadhan123/NLP/blob/main/NLP_lab_14_4_2266.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import libraries

In [17]:

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split

# TEXT DATA

In [33]:
data = {
    'texts': [
        "Win a brand new car",
        "Hello how are you",
        "Please send the report",
        "Free gift voucher available",
        "Let's meet tomorrow",
        "Click here to win prize",
        "I will call you tonight",
        "Exclusive deal just for you",
        "Are you coming to class",
        "Claim your reward now"
    ],
    'labels': [
        1, 0, 0, 1, 0,
        1, 0, 1, 0, 1
    ]
}

# BAG OF WORDS

NUMERICALVECTOR

In [34]:
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(data['texts']).toarray()
y = data['labels']

# Train-test

In [35]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [36]:
class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [37]:
train_loader = DataLoader(TextDataset(X_train, y_train), batch_size=2, shuffle=True)
test_loader = DataLoader(TextDataset(X_test, y_test), batch_size=2)

# CNN Model

In [38]:
# CNN Model
class CNNModel(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        kernels=4
        kernel_size=3
        self.conv = nn.Conv1d(1, kernels, kernel_size)
        pool_size=2
        self.pool = nn.MaxPool1d(pool_size)
        # Corrected calculation for flattened_size
        conv_output_length = input_size - kernel_size + 1
        pool_output_length = (conv_output_length - pool_size) // pool_size + 1
        flattened_size = kernels * pool_output_length
        hidden_neurons=8 # Hidden layer
        self.fc1 = nn.Linear(flattened_size, hidden_neurons)   # 8 hidden layer neurons
        output_neurons=1  # Output layer
        self.fc2 = nn.Linear(hidden_neurons, output_neurons)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        x = x.unsqueeze(1)
        x = torch.relu(self.conv(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)       # Flatten
        x = torch.relu(self.fc1(x))     # Hidden + activation
        x = self.fc2(x)                 # Output layer
        return self.sigmoid(x)

In [39]:
model = CNNModel(input_size=X.shape[1])

# Optimizer

In [40]:
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Training and loss

In [41]:
for epoch in range(10):
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        output = model(X_batch).squeeze()
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 0.6921
Epoch 2, Loss: 0.7621
Epoch 3, Loss: 0.6968
Epoch 4, Loss: 0.5679
Epoch 5, Loss: 0.6954
Epoch 6, Loss: 0.7114
Epoch 7, Loss: 0.7082
Epoch 8, Loss: 0.6886
Epoch 9, Loss: 0.6224
Epoch 10, Loss: 0.4261


# TESTING AND EVOLUTION OF THE MODEL

In [42]:
y_pred = []
y_actual = []

model.eval()
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        output = model(X_batch).squeeze()
        preds = (output > 0.5).int()
        y_pred.extend(preds.tolist())
        y_actual.extend(y_batch.tolist())
y_actual = [int(y) for y in y_actual]
print(y_pred)

[0, 0]


In [43]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
print("\nConfusion Matrix:")
print(confusion_matrix(y_actual, y_pred))
print("\nClassification Report:")
print(classification_report(y_actual, y_pred))
print("\nAccuracy Score:", accuracy_score(y_actual, y_pred))


Confusion Matrix:
[[0 0]
 [2 0]]

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       0.0
           1       0.00      0.00      0.00       2.0

    accuracy                           0.00       2.0
   macro avg       0.00      0.00      0.00       2.0
weighted avg       0.00      0.00      0.00       2.0


Accuracy Score: 0.0


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_